# Title

In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline
import pandas
pandas.options.display.max_rows = 6

In [2]:
from tsdm.datasets import BaseDataset

In [3]:
class USHCN(BaseDataset):
    url = "https://cdiac.ess-dive.lbl.gov/ftp/ushcn_daily/"
# USHCN.download()

# State Codes

In [6]:
# Best viewed with elastic tabstops!
from io import StringIO
STATE_CODES = r"""
ID	Abbr.	State
01	AL	Alabama
02	AZ	Arizona
03	AR	Arkansas
04	CA	California
05	CO	Colorado
06	CT	Connecticut
07	DE	Delaware
08	FL	Florida
09	GA	Georgia
10	ID	Idaho
11	IL	Idaho
12	IN	Indiana
13	IA	Iowa
14	KS	Kansas
15	KY	Kentucky
16	LA	Louisiana
17	ME	Maine
18	MD	Maryland
19	MA	Massachusetts
20	MI	Michigan
21	MN	Minnesota
22	MS	Mississippi
23	MO	Missouri
24	MT	Montana
25	NE	Nebraska
26	NV	Nevada
27	NH	NewHampshire
28	NJ	NewJersey
29	NM	NewMexico
30	NY	NewYork
31	NC	NorthCarolina
32	ND	NorthDakota
33	OH	Ohio
34	OK	Oklahoma
35	OR	Oregon
36	PA	Pennsylvania
37	RI	RhodeIsland
38	SC	SouthCarolina
39	SD	SouthDakota
40	TN	Tennessee
41	TX	Texas
42	UT	Utah
43	VT	Vermont
44	VA	Virginia
45	WA	Washington
46	WV	WestVirginia
47	WI	Wisconsin
48	WY	Wyoming
"""


state_dtypes = {
    "ID": pandas.CategoricalDtype(ordered=True),
    "Abbr.": pandas.CategoricalDtype(ordered=True),
    "State": pandas.StringDtype(),
}
states = pandas.read_csv(StringIO(STATE_CODES), sep="\t", dtype=state_dtypes)

,ID,Abbr.,State
0,01,AL,Alabama
1,02,AZ,Arizona
2,03,AR,Arkansas
...,...,...,...
45,46,WV,WestVirginia
46,47,WI,Wisconsin
47,48,WY,Wyoming


In [9]:
states.to_parquet("a.pq")
states2 = pandas.read_parquet("a.pq")
pandas.testing.assert_frame_equal(states, states2)

# Stations Meta-Data

In [12]:
station_colspecs = {
    "COOP_ID" :     ( 1,  6),
    "LATITUDE" :    ( 8, 15),
    "LONGITUDE" :   (17, 25),
    "ELEVATION" :   (27, 32),
    "STATE" :       (34, 35),
    "NAME" :        (37, 66),
    "COMPONENT_1" : (68, 73),
    "COMPONENT_2" : (75, 80),
    "COMPONENT_3" : (82, 87),
    "UTC_OFFSET" :  (89, 90),
}

# fix colspec to 0-index, half open interval
station_colspecs = {key:(a-1, b) for key, (a,b) in station_colspecs.items()}

station_dtypes = {
    "COOP_ID" : pandas.CategoricalDtype(ordered=True),
    "LATITUDE" : pandas.Float32Dtype(),
    "LONGITUDE" : pandas.Float32Dtype(),
    "ELEVATION" : pandas.Float32Dtype(),
    "STATE" : states.ID.dtype,
    "NAME" : pandas.StringDtype(),
    "COMPONENT_1" :pandas.CategoricalDtype(ordered=True),
    "COMPONENT_2" :pandas.CategoricalDtype(ordered=True),
    "COMPONENT_3" : pandas.CategoricalDtype(ordered=True),
    "UTC_OFFSET" : "timedelta64[h]",
}

station_na_values = {
    "ELEVATION"   : -999.9, 
    "COMPONENT_1" : "------",
    "COMPONENT_2" : "------",
    "COMPONENT_3" : "------",
}

{'ELEVATION': -999.9,
 'COMPONENT_1': '------',
 'COMPONENT_2': '------',
 'COMPONENT_3': '------'}

In [13]:
stations_filename = "ushcn-stations.txt"
stations_filepath = USHCN.rawdata_path.joinpath(stations_filename)
stations = pandas.read_fwf(stations_filepath, na_values=station_na_values, colspecs=list(station_colspecs.values()),  header=0, names=station_colspecs, dtype=station_dtypes)
COOP_IDS = pandas.CategoricalDtype(stations.COOP_ID, ordered=True)
stations.astype({
    "COOP_ID" : COOP_IDS,
    "COMPONENT_1" : COOP_IDS,
    "COMPONENT_2" : COOP_IDS,
    "COMPONENT_3" : COOP_IDS,
})

,COOP_ID,LATITUDE,LONGITUDE,ELEVATION,STATE,NAME,COMPONENT_1,COMPONENT_2,COMPONENT_3,UTC_OFFSET
0,012813,30.5467,-87.880798,7.0,NaN,FAIRHOPE 2 NE,NaN,NaN,NaN,0 days 06:00:00
1,013160,32.834702,-88.134201,38.099998,NaN,GAINESVILLE LOCK,NaN,NaN,NaN,0 days 06:00:00
2,013511,32.701698,-87.580803,67.099998,NaN,GREENSBORO,NaN,NaN,NaN,0 days 06:00:00
...,...,...,...,...,...,...,...,...,...,...
1214,489615,42.1106,-104.949203,1413.699951,NaN,WHEATLAND 4 N,NaN,NaN,NaN,0 days 07:00:00
1215,489770,44.010799,-107.968597,1237.5,NaN,WORLAND,NaN,NaN,NaN,0 days 07:00:00
1216,489905,44.9767,-110.696404,1898.900024,NaN,YELLOWSTONE PK MAMMOTH,NaN,NaN,NaN,0 days 07:00:00


In [20]:
stations2

,COOP_ID,LATITUDE,LONGITUDE,ELEVATION,STATE,NAME,COMPONENT_1,COMPONENT_2,COMPONENT_3,UTC_OFFSET
0,012813,30.5467,-87.880798,7.0,NaN,FAIRHOPE 2 NE,NaN,NaN,NaN,0 days 06:00:00
1,013160,32.834702,-88.134201,38.099998,NaN,GAINESVILLE LOCK,011694,NaN,NaN,0 days 06:00:00
2,013511,32.701698,-87.580803,67.099998,NaN,GREENSBORO,NaN,NaN,NaN,0 days 06:00:00
...,...,...,...,...,...,...,...,...,...,...
1214,489615,42.1106,-104.949203,1413.699951,NaN,WHEATLAND 4 N,NaN,NaN,NaN,0 days 07:00:00
1215,489770,44.010799,-107.968597,1237.5,NaN,WORLAND,NaN,NaN,NaN,0 days 07:00:00
1216,489905,44.9767,-110.696404,1898.900024,NaN,YELLOWSTONE PK MAMMOTH,NaN,NaN,NaN,0 days 07:00:00


In [19]:
stations.to_feather("a.pq")
stations2 = pandas.read_feather("a.pq")
pandas.testing.assert_frame_equal(stations, stations2)

# Station Data

In [9]:
MFLAGS = pandas.CategoricalDtype(("B", "D", "H", "K", "L", "O", "P", "T", "W"))
QFLAGS = pandas.CategoricalDtype(("D", "G", "I", "K", "L", "M", "N", "O", "R", "S", "T", "W", "X", "Z"))
SFLAGS = pandas.CategoricalDtype(("0", "6", "7", "A", "B", "F", "G", "H", "K", "M", "N", "R", "S", "T", "U", "W", "X", "Z"))
ELEMENTS = pandas.CategoricalDtype(("PRCP", "SNOW", "SNWD", "TMAX", "TMIN"))


dtypes = {
        "COOP_ID": COOP_IDS,
        "YEAR" : pandas.UInt16Dtype(),
        "MONTH": pandas.UInt8Dtype(),
        "ELEMENT" : ELEMENTS,
        "VALUE" : pandas.Int16Dtype(),
        "MFLAG" : MFLAGS,
        "QFLAG" : QFLAGS,
        "SFLAG" : SFLAGS,
}

# column start, stop, dtype
colspecs = {
    "COOP_ID" : (1, 6),
    "YEAR":     (7, 10),
    "MONTH":    (11, 12),
    "ELEMENT":  (13, 16),
}

for k, i in enumerate(range(17, 258, 8)):
    colspecs |= {
        ("VALUE", k+1) : (i, i+4),
        ("MFLAG", k+1) : (i+5, i+5),
        ("QFLAG", k+1) : (i+6, i+6),
        ("SFLAG", k+1) : (i+7, i+7),
    }

    # dtype |= {
    #     f"VALUE-{k+1}" : integer,
    #     f"MFLAG-{k+1}" : mflag_types,
    #     f"QFLAG-{k+1}" : qflag_types,
    #     f"SFLAG-{k+1}" : sflag_types,
    # }
    
    
# These should coincide with the description in data_format.txt
widths = [ b - a + 1 for a, b in colspecs.values() ]
dtype = {key:(dtypes[key[0]] if isinstance(key, tuple) else dtypes[key]) for key in colspecs}

cspec = [(a-1,b-1) for a,b in colspecs.values() ]
na_values = {("VALUE", k):-9999 for k in range(1, 32)}
# ds = pandas.read_fwf("state32.txt", names=colspecs, widths=widths, header=None, dtype=dtype, na_values=-9999)

[(0, 5),
 (6, 9),
 (10, 11),
 (12, 15),
 (16, 20),
 (21, 21),
 (22, 22),
 (23, 23),
 (24, 28),
 (29, 29),
 (30, 30),
 (31, 31),
 (32, 36),
 (37, 37),
 (38, 38),
 (39, 39),
 (40, 44),
 (45, 45),
 (46, 46),
 (47, 47),
 (48, 52),
 (53, 53),
 (54, 54),
 (55, 55),
 (56, 60),
 (61, 61),
 (62, 62),
 (63, 63),
 (64, 68),
 (69, 69),
 (70, 70),
 (71, 71),
 (72, 76),
 (77, 77),
 (78, 78),
 (79, 79),
 (80, 84),
 (85, 85),
 (86, 86),
 (87, 87),
 (88, 92),
 (93, 93),
 (94, 94),
 (95, 95),
 (96, 100),
 (101, 101),
 (102, 102),
 (103, 103),
 (104, 108),
 (109, 109),
 (110, 110),
 (111, 111),
 (112, 116),
 (117, 117),
 (118, 118),
 (119, 119),
 (120, 124),
 (125, 125),
 (126, 126),
 (127, 127),
 (128, 132),
 (133, 133),
 (134, 134),
 (135, 135),
 (136, 140),
 (141, 141),
 (142, 142),
 (143, 143),
 (144, 148),
 (149, 149),
 (150, 150),
 (151, 151),
 (152, 156),
 (157, 157),
 (158, 158),
 (159, 159),
 (160, 164),
 (165, 165),
 (166, 166),
 (167, 167),
 (168, 172),
 (173, 173),
 (174, 174),
 (175, 175),
 

In [ ]:
from zipfile import ZipFile
import gzip
fname = "state01_AL.txt"
fpath = USHCN.rawdata_path.joinpath(fname + ".gz")

In [ ]:
%%time
with gzip.open(fpath) as file:
    ds = pandas.read_fwf(file, names=colspecs, widths=widths, header=None, dtype=dtype, na_values=-9999)

ds

In [ ]:
ds[("QFLAG", 1)].fill_na

# preprocessing the data

In [ ]:
id_cols = ['COOP_ID','YEAR','MONTH','ELEMENT']
data_cols =  ["VALUE", "MFLAG", "QFLAG", "SFLAG"]
data_cols = [col for col in ds.columns if col not in id_cols]
columns = pandas.MultiIndex.from_tuples(ds[data_cols], names=["VAR", "DAY"])
data = pandas.DataFrame(ds[data_cols], columns=columns)
data.index.name="INDEX"
data

In [ ]:
%%time
# Pure magic https://stackoverflow.com/a/27044843/9318372
data = data.stack(level="DAY", dropna=False).reset_index(level="DAY")

In [ ]:
%%time
data = ds[id_cols].join(data, how="inner").reset_index()
data = data.astype(dtypes | {"DAY" : integer})
data = data[["COOP_ID","YEAR","MONTH","DAY","ELEMENT","MFLAG","QFLAG","SFLAG","VALUE"]]

In [ ]:
%%time
mask = pandas.isna(data[["MFLAG","QFLAG","SFLAG","VALUE"]]).sum(axis=1) < 4
data = data[mask]
data = data.sort_values(by=["YEAR", "MONTH", "DAY", "COOP_ID", "ELEMENT"]).reset_index(drop=True)
data

# ALternative: Use Modin for speedup

In [1]:
import os

os.environ["MODIN_ENGINE"] = "ray"
from modin import pandas as pd

In [2]:
import sys

In [9]:
{"ray", "modin"} <= sys.modules.keys()

True

In [5]:
import os
import ray

In [11]:
ray.init(num_cpus=os.cpu_count()-2)

os.environ["MODIN_ENGINE"] = "ray"  # Modin will use Ray
# os.environ["MODIN_ENGINE"] = "dask"  # Modin will use Dask

2021-10-02 20:51:05,712	INFO services.py:1263 -- View the Ray dashboard at http://127.0.0.1:8265


In [12]:
# problem: currently only works uncompressed.

from modin import pandas as pd
fname = "us.txt"
fpath2 = USHCN.rawdata_path.joinpath(fname)

PosixPath('/home/rscholz/.tsdm/rawdata/USHCN/us.txt')

In [13]:
%%time
ds = pd.read_fwf(fpath2, names=colspecs, widths=widths, header=None, na_values=-9999, dtype=dtype)

CPU times: user 1.65 s, sys: 1.03 s, total: 2.68 s
Wall time: 30.3 s


In [14]:
id_cols = ['COOP_ID','YEAR','MONTH','ELEMENT']
data_cols =  ["VALUE", "MFLAG", "QFLAG", "SFLAG"]
data_cols = [col for col in ds.columns if col not in id_cols]
columns = pd.MultiIndex.from_tuples(ds[data_cols], names=["VAR", "DAY"])
data = pd.DataFrame(ds[data_cols])
data.columns = columns
data

VAR,VALUE,MFLAG,QFLAG,SFLAG,VALUE,MFLAG,QFLAG,SFLAG,VALUE,MFLAG,...,QFLAG,SFLAG,VALUE,MFLAG,QFLAG,SFLAG,VALUE,MFLAG,QFLAG,SFLAG
DAY,1,1,1,1,2,2,2,2,3,3,...,29,29,30,30,30,30,31,31,31,31
0,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,...,NaN,6,55,NaN,NaN,6,68,NaN,NaN,6
1,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,...,NaN,6,41,NaN,NaN,6,44,NaN,NaN,6
2,0,NaN,NaN,6,0,NaN,NaN,6,0,NaN,...,NaN,6,0,NaN,NaN,6,0,NaN,NaN,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6728236,0,NaN,NaN,7,0,NaN,NaN,7,0,NaN,...,NaN,7,0,T,NaN,7,0,NaN,NaN,7
6728237,0,NaN,NaN,7,0,NaN,NaN,7,0,NaN,...,NaN,7,0,T,NaN,7,0,NaN,NaN,7
6728238,1,NaN,NaN,7,1,NaN,NaN,7,1,NaN,...,NaN,7,7,NaN,NaN,7,7,NaN,NaN,7


In [15]:
%%time
# Pure magic https://stackoverflow.com/a/27044843/9318372
data = data.stack(level="DAY", dropna=True).reset_index(level="DAY")

CPU times: user 6.95 s, sys: 2.69 s, total: 9.64 s
Wall time: 24.8 s


In [16]:
%%time
_dtypes = {k:v for k,v in dtypes.items() if k in data.columns} | {"DAY" : pandas.UInt8Dtype()}
data = data.astype(_dtypes)

CPU times: user 4.59 s, sys: 1.02 s, total: 5.61 s
Wall time: 23.6 s


In [17]:
%%time
data = ds[id_cols].join(data, how="inner")

CPU times: user 9.86 s, sys: 8.31 s, total: 18.2 s
Wall time: 28.7 s


In [18]:
data.info()

<class 'modin.pandas.dataframe.DataFrame'>
Int64Index: 199370091 entries, 0 to 6728238
Data columns (total 9 columns):
 #   Column   Non-Null Count      Dtype   
---  -------  ------------------  -----   
 0   COOP_ID  199215059 non-null  category
 1   YEAR     199370091 non-null  UInt16
 2   MONTH    199370091 non-null  UInt8
 3   ELEMENT  199370091 non-null  category
 4   DAY      199370091 non-null  UInt8
 5   MFLAG    30808643 non-null   category
 6   QFLAG    741634 non-null     category
 7   SFLAG    199370091 non-null  category
 8   VALUE    199370091 non-null  Int16
dtypes: UInt8(2), category(1), UInt16(1), category(1), category(1), category(1), category(1), Int16(1)
memory usage: 4.5 GB


In [19]:
data[["COOP_ID"]].info()

<class 'modin.pandas.dataframe.DataFrame'>
Int64Index: 199370091 entries, 0 to 6728238
Data columns (total 1 columns):
 #   Column   Non-Null Count      Dtype   
---  -------  ------------------  -----   
 0   COOP_ID  199215059 non-null  category
dtypes: category(1)
memory usage: 1.9 GB


In [23]:
data = data.reset_index(drop=True)

,COOP_ID,YEAR,MONTH,ELEMENT,DAY,MFLAG,QFLAG,SFLAG,VALUE
0,NaN,1926,1,TMAX,21,NaN,NaN,6,75
1,NaN,1926,1,TMAX,22,NaN,NaN,6,72
2,NaN,1926,1,TMAX,23,NaN,NaN,6,41
...,...,...,...,...,...,...,...,...,...
199370088,489905,2014,12,SNWD,29,NaN,NaN,7,7
199370089,489905,2014,12,SNWD,30,NaN,NaN,7,7
199370090,489905,2014,12,SNWD,31,NaN,NaN,7,7


In [26]:
data.to_parquet(USHCN.dataset_file)

In [20]:
%%time
data = data[["COOP_ID","YEAR","MONTH","DAY","ELEMENT","MFLAG","QFLAG","SFLAG","VALUE"]]
data = data.sort_values(by=["YEAR", "MONTH", "DAY", "COOP_ID", "ELEMENT"]).reset_index(drop=True)

To request implementation, send an email to feature_requests@modin.org.


CPU times: user 2min 3s, sys: 32.4 s, total: 2min 36s
Wall time: 4min 27s


In [32]:
%%time
df2 = pd.read_feather(USHCN.dataset_path.joinpath("USHCN.feather"))

CPU times: user 4.9 s, sys: 993 ms, total: 5.89 s
Wall time: 6.62 s


In [33]:
%%time
df2 = pd.read_parquet(USHCN.dataset_path.joinpath("USHCN.parquet"))

CPU times: user 372 ms, sys: 75.6 ms, total: 448 ms
Wall time: 4.44 s


In [ ]:
%%time
mask = pandas.isna(data[["MFLAG","QFLAG","SFLAG","VALUE"]]).sum(axis=1) < 4
data = data[mask]
data